# QC: Meaningful row-match Excel vs Lake — Jan–Jun 2026

## Зачем
Нужен **построчный** % совпадения основных колонок на ключах `(inn, agr)`,  
**без** строк, где значение `NaN` / `None` / `0` / `0.0` (их много и они раздувают %).

Главный показатель: **`both_meaningful_exact_pct`** — exact match среди строк, где **обе** стороны Meaningful.

## Месяцы
`2026-01` … `2026-06` (все Excel-референсы из боевого `01_07_acq_dash_jan_jun_mpos`).

## Колонки
`retl_cnt`, `term_cnt`, `trx_cnt`, `trx_sum`, `commission_from_ops`, `commission_monthly`,  
`aur`, `amortization`, `fin_result`.

## Входы
- Lake: `final_df_period_2026_01_2026_06_mpos.csv` или checkpoints
- Excel: `01_Январь_2026.xlsx` … `06_Июнь_2026.xlsx`

Выходы: `/home/jovyan/documents/Equaring/Data/debug_trx_may_2026/`


In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)
pd.set_option('display.max_colwidth', 80)

MONTHS = ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']
# optional deep-dive month (TOP mismatches etc.)
TARGET_MONTH = '2026-05'

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')

excel_by_month = {
    '2026-01': DATA_DIR / '01_Январь_2026.xlsx',
    '2026-02': DATA_DIR / '02_Февраль_2026.xlsx',
    '2026-03': DATA_DIR / '03_Март_2026.xlsx',
    '2026-04': DATA_DIR / '04_Апрель_2026.xlsx',
    '2026-05': DATA_DIR / '05_Май_2026.xlsx',
    '2026-06': DATA_DIR / '06_Июнь_2026.xlsx',
}
excel_header_by_month = {
    '2026-01': 1,
    '2026-02': 1,
}

period_csv = DATA_DIR / 'final_df_period_2026_01_2026_06_mpos.csv'
checkpoint_dir = DATA_DIR / 'checkpoints_final_df_2026_01_2026_06_mpos'

OUT_DIR = DATA_DIR / 'debug_trx_may_2026'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('MONTHS =', MONTHS)
print('TARGET_MONTH (deep-dive) =', TARGET_MONTH)
print('period_csv exists =', period_csv.exists(), period_csv)
print('checkpoint_dir exists =', checkpoint_dir.exists(), checkpoint_dir)
for m in MONTHS:
    p = excel_by_month[m]
    h = excel_header_by_month.get(m, 0)
    print(f'Excel {m}: exists={p.exists()} | header={h} | {p}')


## Helpers


In [ ]:
def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace('\xa0', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce'
    )


def pick_col_robust(columns, candidates):
    cols = list(columns)
    norm = lambda x: re.sub(r'\s+', ' ', str(x).replace('\xa0', ' ').strip().lower())
    norm_map = {norm(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        nc = norm(c)
        if nc in norm_map:
            return norm_map[nc]
    return None


def is_meaningful(s):
    v = pd.to_numeric(s, errors='coerce')
    return v.notna() & (v != 0)


COL_MAP = {
    'inn_col': ['ИНН', 'inn', 'c_inn'],
    'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
    'retl_col': ['Кол-во торговых точек', 'Ко-во торговых точек', 'Количество торговых точек'],
    'term_col': ['Кол-во терминалов', 'Количество терминалов'],
    'trx_cnt_col': ['Количество операций', 'Количеств операций', 'trx_cnt'],
    'trx_sum_col': ['Сумма операций', 'Сумма опреаций', 'trx_sum'],
    'comm_ops_col': [
        'Комиссия эквайринга',
        'Комиссия (% с операций)',
        'Комиссия \n(% с операций)',
        'Комиссия % с операций',
    ],
    'comm_monthly_col': [
        'Комиссия в месяц',
        'Комиссия CN (₽ в месяц)',
        'Комиссия (₽ в месяц)',
        'Комиссия \n(₽ в месяц)',
        'Комиссия (руб в месяц)',
    ],
    'aur_col': ['АУР', 'AUR', 'Aur', 'Аур'],
    'amortization_col': [
        'Амортизация', 'Аморт', 'Амортизация терминалов',
        'amortization', 'Amortization', 'Амортизация, руб',
    ],
    'fin_result_col': [
        'Фин. Рез.', 'Фин.Рез.', 'Фин.рез.', 'Фин. рез.',
        'Фин рез', 'Финрез', 'Фин результат', 'Финансовый результат',
        'fin_result', 'Fin.Res.', 'FinRes',
    ],
}

# metric, lake_col, excel_col, tol
METRICS_ALL = [
    ('retl_cnt', 'retl_cnt_lake', 'retl_cnt_excel', 0.0),
    ('term_cnt', 'term_cnt_lake', 'term_cnt_excel', 0.0),
    ('trx_cnt', 'trx_cnt_lake', 'trx_cnt_excel', 0.0),
    ('trx_sum', 'trx_sum_lake', 'trx_sum_excel', 0.01),
    ('commission_from_ops', 'commission_from_ops_lake', 'commission_from_ops_excel', 0.01),
    ('commission_monthly', 'commission_monthly_lake', 'commission_monthly_excel', 0.01),
    ('aur', 'aur_lake', 'aur_excel', 0.01),
    ('amortization', 'amortization_lake', 'amortization_excel', 0.01),
    ('fin_result', 'fin_result_lake', 'fin_result_excel', 0.01),
]

LAKE_NUM_COLS = [
    'retl_cnt', 'term_cnt', 'trx_cnt', 'trx_sum',
    'commission_from_ops', 'commission_monthly',
    'aur', 'amortization', 'fin_result',
]


## Loaders: lake + Excel


In [ ]:
def load_final_df_month(month_label: str) -> pd.DataFrame:
    if period_csv.exists():
        df = pd.read_csv(period_csv, dtype={'inn': 'string', 'agr_id': 'string', 'n_agr': 'string'})
        if 'report_month' not in df.columns:
            raise RuntimeError(f'no report_month in {period_csv}')
        out = df.loc[df['report_month'].astype(str) == month_label].copy()
        if len(out):
            print(f'lake from period CSV: {month_label} rows={len(out):,}')
            return out
        print(f'period CSV has 0 rows for {month_label}, try checkpoint')

    month_us = month_label.replace('-', '_')
    candidates = []
    if checkpoint_dir.exists():
        candidates += sorted(checkpoint_dir.glob(f'*{month_us}*'))
        candidates += sorted(checkpoint_dir.glob(f'*{month_label}*'))
        for pat in [
            f'final_df_{month_label}.csv',
            f'final_df_{month_label}.parquet',
            f'{month_label}.csv',
            f'{month_label}.parquet',
        ]:
            p = checkpoint_dir / pat
            if p.exists():
                candidates = [p] + candidates
        if not candidates:
            print('checkpoint files:', [p.name for p in sorted(checkpoint_dir.iterdir())[:40]])

    for p in candidates:
        if not p.exists() or not p.is_file():
            continue
        if p.suffix.lower() == '.csv':
            out = pd.read_csv(p, dtype={'inn': 'string', 'agr_id': 'string', 'n_agr': 'string'})
        elif p.suffix.lower() == '.parquet':
            out = pd.read_parquet(p)
        else:
            continue
        if 'report_month' in out.columns:
            out = out.loc[out['report_month'].astype(str) == month_label].copy()
        print(f'lake from checkpoint: {p.name} rows={len(out):,}')
        return out

    raise FileNotFoundError(
        f'Не найден final_df за {month_label}. Проверь {period_csv} или {checkpoint_dir}'
    )


def build_lake_agg(final_df: pd.DataFrame) -> pd.DataFrame:
    lk = final_df.copy()
    if 'agr_id' not in lk.columns and 'n_agr' in lk.columns:
        lk['agr_id'] = lk['n_agr']
    lk['inn_key'] = lk['inn'].apply(normalize_inn_q1)
    lk['agr_id_key'] = lk['agr_id'].apply(normalize_agr_q1)

    for c in LAKE_NUM_COLS:
        if c not in lk.columns:
            lk[c] = np.nan
        lk[c] = pd.to_numeric(lk[c], errors='coerce')

    return (
        lk.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg(
              retl_cnt_lake=('retl_cnt', 'max'),
              term_cnt_lake=('term_cnt', 'max'),
              trx_cnt_lake=('trx_cnt', 'max'),
              trx_sum_lake=('trx_sum', 'max'),
              commission_from_ops_lake=('commission_from_ops', 'max'),
              commission_monthly_lake=('commission_monthly', 'max'),
              aur_lake=('aur', 'max'),
              amortization_lake=('amortization', 'max'),
              fin_result_lake=('fin_result', 'max'),
              rows_lake=('trx_cnt', 'size'),
          )
    )


def load_excel_raw(month_label: str):
    path = excel_by_month[month_label]
    header = int(excel_header_by_month.get(month_label, 0))
    if not path.exists():
        raise FileNotFoundError(path)
    ex = pd.read_excel(path, header=header)
    resolved = {k: pick_col_robust(ex.columns, v) for k, v in COL_MAP.items()}
    print(f'Excel {month_label}: header={header} shape={ex.shape}')
    missing_required = [
        k for k in [
            'inn_col', 'agr_col', 'retl_col', 'term_col',
            'trx_cnt_col', 'trx_sum_col', 'comm_ops_col', 'comm_monthly_col',
        ] if resolved.get(k) is None
    ]
    if missing_required:
        raise ValueError(f'{month_label}: missing Excel cols {missing_required}; available={list(ex.columns)}')
    return ex, resolved, header


def enrich_excel(ex, resolved):
    out = ex.copy()
    out['inn_key'] = out[resolved['inn_col']].apply(normalize_inn_q1)
    out['agr_id_key'] = out[resolved['agr_col']].apply(normalize_agr_q1)
    out['retl_cnt_excel'] = pd.to_numeric(out[resolved['retl_col']], errors='coerce')
    out['term_cnt_excel'] = pd.to_numeric(out[resolved['term_col']], errors='coerce')
    out['trx_cnt_excel'] = pd.to_numeric(out[resolved['trx_cnt_col']], errors='coerce')
    out['trx_sum_excel'] = to_num_series(out[resolved['trx_sum_col']])
    out['commission_from_ops_excel'] = to_num_series(out[resolved['comm_ops_col']])
    out['commission_monthly_excel'] = to_num_series(out[resolved['comm_monthly_col']])
    if resolved.get('aur_col') is not None:
        out['aur_excel'] = to_num_series(out[resolved['aur_col']])
    else:
        out['aur_excel'] = np.nan
    if resolved.get('amortization_col') is not None:
        out['amortization_excel'] = to_num_series(out[resolved['amortization_col']])
    else:
        out['amortization_excel'] = np.nan
    if resolved.get('fin_result_col') is not None:
        out['fin_result_excel'] = to_num_series(out[resolved['fin_result_col']])
    else:
        out['fin_result_excel'] = np.nan
    return out


def build_excel_agg(ex_enr: pd.DataFrame) -> pd.DataFrame:
    # как в боевом compare: trx_sum/ops/amort/fin = sum; counts/monthly/aur = max
    return (
        ex_enr.dropna(subset=['inn_key', 'agr_id_key'])
        .groupby(['inn_key', 'agr_id_key'], as_index=False)
        .agg(
            retl_cnt_excel=('retl_cnt_excel', 'max'),
            term_cnt_excel=('term_cnt_excel', 'max'),
            trx_cnt_excel=('trx_cnt_excel', 'max'),
            trx_sum_excel=('trx_sum_excel', 'sum'),
            commission_from_ops_excel=('commission_from_ops_excel', 'sum'),
            commission_monthly_excel=('commission_monthly_excel', 'max'),
            aur_excel=('aur_excel', 'max'),
            amortization_excel=('amortization_excel', 'sum'),
            fin_result_excel=('fin_result_excel', 'sum'),
        )
    )


print('loaders ready')


## Главная проверка: Meaningful row-match по 6 месяцам

- **meaningful / both_meaningful_exact_pct** — основной ответ (обе стороны не 0/NaN).
- **battle_fillna0** — как старый row match (`fillna(0)`, знаменатель = все both).


In [ ]:
def row_match_report(both_df, metrics, label, mode='meaningful'):
    rows = []
    n_both = len(both_df)
    print(f'\n===== ROW MATCH {label} | mode={mode} | keys_both={n_both:,} =====')
    for name, lc, ec, tol in metrics:
        if lc not in both_df.columns or ec not in both_df.columns:
            continue
        lv_raw = pd.to_numeric(both_df[lc], errors='coerce')
        ev_raw = pd.to_numeric(both_df[ec], errors='coerce')

        # skip metric if Excel column entirely empty
        if ev_raw.notna().sum() == 0:
            print(f'  {name}: skip (no Excel values)')
            continue

        if mode == 'battle_fillna0':
            lv = lv_raw.fillna(0.0)
            ev = ev_raw.fillna(0.0)
            denom_mask = pd.Series(True, index=both_df.index)
        else:
            denom_mask = is_meaningful(lv_raw) | is_meaningful(ev_raw)
            lv = lv_raw
            ev = ev_raw

        denom = int(denom_mask.sum())
        if denom == 0:
            print(f'  {name}: no rows in denom')
            continue

        if tol and tol > 0:
            exact = pd.Series(
                np.isclose(
                    lv.to_numpy(dtype=float, na_value=np.nan),
                    ev.to_numpy(dtype=float, na_value=np.nan),
                    atol=tol,
                    rtol=0,
                    equal_nan=True,
                ),
                index=both_df.index,
            )
        else:
            if mode == 'battle_fillna0':
                exact = lv.eq(ev)
            else:
                exact = (lv == ev) | (lv.isna() & ev.isna())

        exact_n = int((exact & denom_mask).sum())
        mismatch_n = denom - exact_n
        exact_pct = round(100.0 * exact_n / denom, 2)

        both_m = is_meaningful(lv_raw) & is_meaningful(ev_raw)
        both_m_n = int(both_m.sum())
        both_m_exact = int((exact & both_m).sum()) if both_m_n else 0
        both_m_pct = round(100.0 * both_m_exact / both_m_n, 2) if both_m_n else None

        rows.append({
            'report_month': label,
            'mode': mode,
            'metric': name,
            'keys_both': n_both,
            'denom_n': denom,
            'exact_n': exact_n,
            'exact_pct': exact_pct,
            'mismatch_n': mismatch_n,
            'both_meaningful_n': both_m_n,
            'both_meaningful_exact_n': both_m_exact,
            'both_meaningful_exact_pct': both_m_pct,
            'both_meaningful_share_of_both': round(100.0 * both_m_n / n_both, 2) if n_both else None,
            'tol': tol,
        })
        print(
            f"  {name}: denom_exact={exact_pct}% ({exact_n}/{denom}), mismatch={mismatch_n}"
            f" | both_meaningful exact={both_m_pct}% ({both_m_exact}/{both_m_n})"
            f" | share_both={rows[-1]['both_meaningful_share_of_both']}%"
        )
    return pd.DataFrame(rows)


def run_month(month_label: str) -> pd.DataFrame:
    lake_raw = load_final_df_month(month_label)
    lake_agg = build_lake_agg(lake_raw)
    ex_raw, resolved, _ = load_excel_raw(month_label)
    ex_enr = enrich_excel(ex_raw, resolved)
    ex_agg = build_excel_agg(ex_enr)

    cmp = lake_agg.merge(ex_agg, on=['inn_key', 'agr_id_key'], how='outer', indicator=True)
    both = cmp.loc[cmp['_merge'] == 'both'].copy()
    print(
        f'{month_label}: keys both={len(both):,} | only_lake={(cmp["_merge"]=="left_only").sum():,}'
        f' | only_excel={(cmp["_merge"]=="right_only").sum():,}'
    )
    parts = [
        row_match_report(both, METRICS_ALL, month_label, mode='battle_fillna0'),
        row_match_report(both, METRICS_ALL, month_label, mode='meaningful'),
    ]
    return pd.concat(parts, ignore_index=True)


all_parts = []
errors = []
for month in MONTHS:
    print('\n' + '=' * 72)
    print('MONTH', month)
    print('=' * 72)
    try:
        all_parts.append(run_month(month))
    except Exception as exc:
        errors.append({'report_month': month, 'error': f'{type(exc).__name__}: {exc}'})
        print('ERROR', month, errors[-1]['error'])

meaningful_row_match_df = pd.concat(all_parts, ignore_index=True) if all_parts else pd.DataFrame()
if errors:
    display(pd.DataFrame(errors))

out_long = OUT_DIR / 'meaningful_row_match_2026_01_06.csv'
meaningful_row_match_df.to_csv(out_long, index=False, encoding='utf-8-sig')
print('\nSaved long:', out_long)
display(meaningful_row_match_df.head(30))


## Сводка: pivot `both_meaningful_exact_pct` (главный ответ)


In [ ]:
if meaningful_row_match_df.empty:
    raise RuntimeError('meaningful_row_match_df empty — fix errors above')

mm = meaningful_row_match_df.loc[meaningful_row_match_df['mode'] == 'meaningful'].copy()

pivot_pct = mm.pivot_table(
    index='report_month',
    columns='metric',
    values='both_meaningful_exact_pct',
    aggfunc='first',
)
metric_order = [m for m, *_ in METRICS_ALL if m in pivot_pct.columns]
pivot_pct = pivot_pct.reindex(columns=metric_order)

pivot_n = mm.pivot_table(
    index='report_month',
    columns='metric',
    values='both_meaningful_n',
    aggfunc='first',
).reindex(columns=metric_order)

pivot_share = mm.pivot_table(
    index='report_month',
    columns='metric',
    values='both_meaningful_share_of_both',
    aggfunc='first',
).reindex(columns=metric_order)

# battle fillna0 for comparison with old screenshots
bb = meaningful_row_match_df.loc[meaningful_row_match_df['mode'] == 'battle_fillna0'].copy()
pivot_battle = bb.pivot_table(
    index='report_month',
    columns='metric',
    values='exact_pct',
    aggfunc='first',
).reindex(columns=metric_order)

print('=== MAIN: both_meaningful_exact_pct (row match on non-zero both sides) ===')
display(pivot_pct)

print('=== both_meaningful_n (denominator size) ===')
display(pivot_n)

print('=== share of keys_both that are both_meaningful (%) ===')
display(pivot_share)

print('=== REF: battle_fillna0 exact_pct (old row match style) ===')
display(pivot_battle)

pivot_pct.to_csv(OUT_DIR / 'meaningful_exact_pct_pivot_2026_01_06.csv', encoding='utf-8-sig')
pivot_n.to_csv(OUT_DIR / 'meaningful_n_pivot_2026_01_06.csv', encoding='utf-8-sig')
pivot_battle.to_csv(OUT_DIR / 'battle_fillna0_exact_pct_pivot_2026_01_06.csv', encoding='utf-8-sig')
print('Saved pivots to', OUT_DIR)

# quick highlight: worst months for trx_cnt
if 'trx_cnt' in pivot_pct.columns:
    s = pivot_pct['trx_cnt'].sort_values()
    print('\ntrx_cnt both_meaningful_exact_pct by month (asc):')
    display(s.to_frame('trx_cnt_both_meaningful_exact_pct'))


## Deep-dive (опционально): май — TOP mismatches

Секции ниже не обязательны для ответа по 6 месяцам. Используй, если май просел.


In [ ]:
# Deep-dive May: rebuild both for TARGET_MONTH and export TOP mismatches
lake_raw_t = load_final_df_month(TARGET_MONTH)
lake_agg_t = build_lake_agg(lake_raw_t)
ex_raw_t, res_t, _ = load_excel_raw(TARGET_MONTH)
ex_agg_t = build_excel_agg(enrich_excel(ex_raw_t, res_t))
cmp_t = lake_agg_t.merge(ex_agg_t, on=['inn_key', 'agr_id_key'], how='outer', indicator=True)
both_t = cmp_t.loc[cmp_t['_merge'] == 'both'].copy()

for metric, lc, ec in [
    ('trx_cnt', 'trx_cnt_lake', 'trx_cnt_excel'),
    ('trx_sum', 'trx_sum_lake', 'trx_sum_excel'),
    ('commission_from_ops', 'commission_from_ops_lake', 'commission_from_ops_excel'),
]:
    a = pd.to_numeric(both_t[lc], errors='coerce')
    e = pd.to_numeric(both_t[ec], errors='coerce')
    both_t[f'{metric}_delta'] = a - e
    both_t[f'{metric}_abs_delta'] = (a - e).abs()
    both_t[f'{metric}_ratio'] = np.where(e.fillna(0) == 0, np.nan, a / e)

mask = is_meaningful(both_t['trx_cnt_lake']) & is_meaningful(both_t['trx_cnt_excel'])
mis = both_t.loc[mask & (both_t['trx_cnt_lake'] != both_t['trx_cnt_excel'])].copy()
cols_show = [
    'inn_key', 'agr_id_key',
    'trx_cnt_lake', 'trx_cnt_excel', 'trx_cnt_delta', 'trx_cnt_ratio',
    'trx_sum_lake', 'trx_sum_excel', 'trx_sum_ratio',
    'commission_from_ops_lake', 'commission_from_ops_excel',
    'commission_monthly_lake', 'commission_monthly_excel',
]
print(f'{TARGET_MONTH}: both_meaningful trx_cnt mismatches =', f'{len(mis):,}')
display(mis.sort_values('trx_cnt_abs_delta', ascending=False)[cols_show].head(30))
mis.sort_values('trx_cnt_abs_delta', ascending=False)[cols_show].head(200).to_csv(
    OUT_DIR / f'top_trx_mismatch_meaningful_{TARGET_MONTH}.csv', index=False, encoding='utf-8-sig'
)
print('saved', OUT_DIR / f'top_trx_mismatch_meaningful_{TARGET_MONTH}.csv')
